In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
DATA_PATH = Path("../data/raw/cars.csv")

print("Dataset exists:", DATA_PATH.exists())
print(f"File size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

Dataset exists: True
File size: 138.49 MB


In [3]:
df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 762091
Columns: 20


In [4]:
df.head()

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0
1,Acura,ILX Hybrid 1.5L,2013,47645.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Gray,Ebony,1.0,1.0,1.0,Kars Today,NaN,4.4,12.0,NaN,17995.0
2,Acura,ILX Hybrid 1.5L,2013,53422.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Bellanova White Pearl,Ebony,0.0,1.0,1.0,Weiss Toyota of South County,4.3,4.4,12.0,500.0,17000.0
3,Acura,ILX Hybrid 1.5L,2013,117598.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Polished Metal Metallic,NaN,0.0,1.0,1.0,Apple Tree Acura,NaN,4.4,12.0,675.0,14958.0
4,Acura,ILX Hybrid 1.5L,2013,114865.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,NaN,Ebony,1.0,0.0,1.0,Herb Connolly Chevrolet,3.7,4.4,12.0,300.0,14498.0


In [5]:
df.columns.tolist()

['manufacturer',
 'model',
 'year',
 'mileage',
 'engine',
 'transmission',
 'drivetrain',
 'fuel_type',
 'mpg',
 'exterior_color',
 'interior_color',
 'accidents_or_damage',
 'one_owner',
 'personal_use_only',
 'seller_name',
 'seller_rating',
 'driver_rating',
 'driver_reviews_num',
 'price_drop',
 'price']

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 762091 entries, 0 to 762090
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   manufacturer         762091 non-null  str    
 1   model                762091 non-null  str    
 2   year                 762091 non-null  int64  
 3   mileage              761585 non-null  float64
 4   engine               747041 non-null  str    
 5   transmission         752187 non-null  str    
 6   drivetrain           740529 non-null  str    
 7   fuel_type            739164 non-null  str    
 8   mpg                  620020 non-null  str    
 9   exterior_color       753232 non-null  str    
 10  interior_color       705116 non-null  str    
 11  accidents_or_damage  737879 non-null  float64
 12  one_owner            730608 non-null  float64
 13  personal_use_only    737239 non-null  float64
 14  seller_name          753498 non-null  str    
 15  seller_rating        548118 

In [7]:
missing_pct = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing_pct

price_drop             46.185954
seller_rating          28.077093
mpg                    18.642262
interior_color          7.476141
driver_rating           4.150685
one_owner               4.131134
personal_use_only       3.261028
accidents_or_damage     3.177048
fuel_type               3.008433
drivetrain              2.829321
engine                  1.974830
transmission            1.299582
exterior_color          1.162460
seller_name             1.127556
mileage                 0.066396
driver_reviews_num      0.000000
manufacturer            0.000000
model                   0.000000
year                    0.000000
price                   0.000000
dtype: float64

## Manufacturer and Model Coverage

The next step is to determine whether the dataset contains enough listings for the enthusiast and performance vehicles targeted by Apex Analytics.

In [8]:
manufacturer_counts = df["manufacturer"].value_counts()

manufacturer_counts.head(50)

manufacturer
Ford             79526
Toyota           59535
Chevrolet        56043
Nissan           48529
Jeep             41665
Mercedes-Benz    40824
Honda            37612
BMW              37570
Kia              35063
GMC              29563
Dodge            25250
Subaru           24767
Volkswagen       24620
Hyundai          22203
Lexus            21301
RAM              19364
Audi             17863
Cadillac         17794
Mazda            15485
Buick            14624
Chrysler         12647
INFINITI         12289
Land Rover       12272
Porsche          11461
Lincoln          10608
Volvo            10029
Acura             8489
Tesla             5883
Mitsubishi        5743
Jaguar            3469
Name: count, dtype: int64

In [9]:
candidate_brands = [
    "BMW",
    "Mercedes-Benz",
    "Porsche",
    "Audi",
    "Chevrolet",
    "Toyota",
    "Nissan",
    "Cadillac"
]

df[df["manufacturer"].isin(candidate_brands)]["manufacturer"].value_counts()

manufacturer
Toyota           59535
Chevrolet        56043
Nissan           48529
Mercedes-Benz    40824
BMW              37570
Audi             17863
Cadillac         17794
Porsche          11461
Name: count, dtype: int64

## BMW Performance Model Coverage

Inspect BMW model naming conventions and determine the number of usable listings for M2, M3, M4, and M5 models.

In [10]:
bmw_models = (
    df.loc[df["manufacturer"] == "BMW", "model"]
      .value_counts()
)

bmw_models.head(100)

model
X3 xDrive30i                             1481
330 i xDrive                             1476
X5 xDrive40i                             1391
330 i                                    1271
X5 xDrive35i                              972
                                         ... 
428 Gran Coupe i xDrive                    85
228 Gran Coupe 228i sDrive Gran Coupe      82
530e 530e xDrive                           82
650 Gran Coupe i xDrive                    82
i3 94 Ah w/Range Extender                  81
Name: count, Length: 100, dtype: int64

In [11]:
bmw_performance = df[
    (df["manufacturer"] == "BMW") &
    (
        df["model"]
        .str.contains(r"\b(M2|M3|M4|M5)\b", case=False, na=False, regex=True)
    )
].copy()

bmw_performance["apex_model"] = (
    bmw_performance["model"]
    .str.extract(r"\b(M2|M3|M4|M5)\b", expand=False)
    .str.upper()
)

bmw_performance["apex_model"].value_counts()

/var/folders/bz/54m5dhc13mz38tytyd8_bmbm0000gn/T/ipykernel_3020/1462650962.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(r"\b(M2|M3|M4|M5)\b", case=False, na=False, regex=True)


apex_model
M4    565
M3    528
M5    396
M2    157
Name: count, dtype: int64

In [12]:
bmw_year_counts = (
    bmw_performance
    .groupby(["apex_model", "year"])
    .size()
    .unstack(fill_value=0)
)

bmw_year_counts

year,1988,1990,1991,1993,1995,1996,1997,1998,1999,2000,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
apex_model,,,,,,,,,,,,,,,,,,,,,
M2,0,0,0,0,0,0,0,0,0,0,...,0,0,7,33,38,9,47,23,0,0
M3,2,2,1,0,3,2,8,7,12,0,...,0,28,30,31,57,0,0,43,60,36
M4,0,0,0,0,0,0,0,0,0,0,...,0,66,79,28,51,39,86,43,131,42
M5,3,0,2,2,0,0,0,0,0,2,...,13,16,12,0,34,96,72,38,26,13


In [13]:
bmw_summary = (
    bmw_performance
    .groupby("apex_model")
    .agg(
        listings=("model", "size"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_price=("price", "median"),
        median_mileage=("mileage", "median")
    )
    .sort_index()
)

bmw_summary

,listings,earliest_year,latest_year,median_price,median_mileage
apex_model,,,,,
M2,157,2016,2021,51999.0,23600.0
M3,528,1988,2023,48900.0,50470.0
M4,565,2015,2023,63677.0,20008.0
M5,396,1988,2023,73934.0,32931.5


In [14]:
pd.crosstab(
    bmw_performance["apex_model"],
    bmw_performance["transmission"],
    margins=True
)

transmission,5 Speed Manual,5-SPEED M/T,5-Speed A/T,5-Speed Automatic,5-Speed M/T,5-Speed Manual,6-SPEED M/T,6-Speed,6-Speed Automatic,6-Speed Automatic with Auto-Shift,...,"Automatic, 8-Spd M STEPTRONIC w/Drivelogic, Sport & Manual Modes",M/T,Manual,Manual 5-Speed,Manual 6-Speed,"Manual, 6-Spd","Manual, 6-Spd w/Overdrive",SMG,Transmission w/Dual Shift Mode,All
apex_model,,,,,,,,,,,,,,,,,,,,,
M2,0,0,0,0,0,0,0,0,2,0,...,0,2,9,0,0,0,0,0,1,146
M3,2,1,2,4,4,15,4,3,1,19,...,0,1,33,1,1,1,1,1,9,516
M4,0,0,0,0,0,0,2,0,1,1,...,3,0,18,0,0,0,0,0,9,548
M5,0,0,0,0,0,4,0,0,0,0,...,0,1,6,0,0,0,0,0,9,379
All,2,1,2,4,4,19,6,3,4,20,...,3,4,66,1,1,1,1,1,28,1589


## Transmission Normalization

The raw dataset contains many inconsistent transmission descriptions. These values are grouped into broader categories to support reliable comparisons between manual and automatic vehicles.

In [15]:
def classify_transmission(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).lower()

    # Automatic-type transmissions first because some descriptions
    # include phrases such as "manual mode"
    automatic_terms = [
        "automatic",
        "a/t",
        "cvt",
        "dct",
        "dual clutch",
        "auto-shift",
        "steptronic",
        "tiptronic"
    ]

    manual_terms = [
        "manual",
        "m/t"
    ]

    if any(term in text for term in automatic_terms):
        return "Automatic"

    if any(term in text for term in manual_terms):
        return "Manual"

    return "Other/Unknown"

In [16]:
bmw_performance["transmission_group"] = (
    bmw_performance["transmission"]
    .apply(classify_transmission)
)

In [17]:
pd.crosstab(
    bmw_performance["apex_model"],
    bmw_performance["transmission_group"],
    margins=True
)

transmission_group,Automatic,Manual,Other/Unknown,Unknown,All
apex_model,,,,,
M2,89,56,1,11,157
M3,280,222,14,12,528
M4,435,101,12,17,565
M5,338,31,10,17,396
All,1142,410,37,57,1646


In [18]:
pd.crosstab(
    bmw_performance["year"],
    bmw_performance["transmission_group"]
)

transmission_group,Automatic,Manual,Other/Unknown,Unknown
year,,,,
1988,0,5,0,0
1990,0,2,0,0
1991,0,3,0,0
1993,0,2,0,0
1995,0,3,0,0
1996,0,2,0,0
1997,2,6,0,0
1998,4,3,0,0
1999,3,9,0,0


## Mercedes-AMG Performance Model Coverage

Inspect Mercedes-Benz model naming conventions and determine whether the dataset contains sufficient C63, E63, and AMG GT listings for analysis.

In [19]:
mercedes_models = (
    df.loc[df["manufacturer"] == "Mercedes-Benz", "model"]
      .value_counts()
)

mercedes_models.head(100)

model
GLC 300 Base 4MATIC        2718
C-Class C 300              1904
C-Class C 300 4MATIC       1633
GLC 300 Base               1526
GLE 350 Base 4MATIC        1464
                           ... 
SL-Class SL550 Roadster      85
AMG GT C                     83
E-Class E 550                83
GLA 250                      83
EQS 580 Base 4MATIC          83
Name: count, Length: 100, dtype: int64

In [20]:
mercedes_amg_names = (
    df.loc[
        (df["manufacturer"] == "Mercedes-Benz") &
        (df["model"].str.contains("AMG", case=False, na=False)),
        "model"
    ]
    .value_counts()
)

mercedes_amg_names.head(100)

model
AMG C 43 Base 4MATIC         432
AMG G 63 Base                409
AMG GLE 53 Base              352
AMG GLC 43 Base 4MATIC       270
AMG C 63 S                   198
                            ... 
G-Class G 55 AMG 4MATIC        9
C-Class AMG C 63               9
AMG CLS 63 S-Model 4MATIC      9
AMG G 63 4x4 Squared           8
AMG CLS 53 4MATIC              8
Name: count, Length: 100, dtype: int64

In [21]:
target_amg_names = (
    df.loc[
        (df["manufacturer"] == "Mercedes-Benz") &
        (
            df["model"].str.contains(
                r"C[\s-]?63|E[\s-]?63|\bAMG GT\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

target_amg_names.head(100)

model
AMG C 63 S                                                   198
AMG GT 53 Base                                               130
AMG E 63 S 4MATIC                                            115
AMG GT C                                                      83
AMG GLE 63 S-Model 4MATIC                                     80
AMG GT 63 S 4-Door                                            69
AMG C 63 Base                                                 69
AMG GT 63 4-Door                                              65
AMG GT Base                                                   56
AMG GT R                                                      50
E-Class E 63 AMG S-Model 4MATIC                               49
AMG GT AMG GT S                                               47
AMG GLC 63 Base 4MATIC                                        43
AMG GT 43 Base                                                35
AMG GLE 63 S 4MATIC Coupe                                     34
E-Class E 63 AMG   

In [22]:
mercedes_amg_names.head(100)

model
AMG C 43 Base 4MATIC         432
AMG G 63 Base                409
AMG GLE 53 Base              352
AMG GLC 43 Base 4MATIC       270
AMG C 63 S                   198
                            ... 
G-Class G 55 AMG 4MATIC        9
C-Class AMG C 63               9
AMG CLS 63 S-Model 4MATIC      9
AMG G 63 4x4 Squared           8
AMG CLS 53 4MATIC              8
Name: count, Length: 100, dtype: int64

In [23]:
target_amg_names.head(100)

model
AMG C 63 S                                                   198
AMG GT 53 Base                                               130
AMG E 63 S 4MATIC                                            115
AMG GT C                                                      83
AMG GLE 63 S-Model 4MATIC                                     80
AMG GT 63 S 4-Door                                            69
AMG C 63 Base                                                 69
AMG GT 63 4-Door                                              65
AMG GT Base                                                   56
AMG GT R                                                      50
E-Class E 63 AMG S-Model 4MATIC                               49
AMG GT AMG GT S                                               47
AMG GLC 63 Base 4MATIC                                        43
AMG GT 43 Base                                                35
AMG GLE 63 S 4MATIC Coupe                                     34
E-Class E 63 AMG   

## Mercedes-AMG Model Normalization

Mercedes-Benz listings use several naming conventions for the same performance models. The following logic normalizes C63, E63, AMG GT two-door, and AMG GT four-door listings while avoiding unrelated models such as GLC 63 and GLE 63.

In [24]:
mercedes = df[df["manufacturer"] == "Mercedes-Benz"].copy()

model_text = mercedes["model"].str.upper()

mercedes["apex_model"] = pd.NA

# C63 family
mercedes.loc[
    model_text.str.contains(r"\bC\s*63\b", regex=True, na=False),
    "apex_model"
] = "C63"

# E63 family
mercedes.loc[
    model_text.str.contains(r"\bE\s*63\b", regex=True, na=False),
    "apex_model"
] = "E63"

# AMG GT 4-Door family
gt_four_door = model_text.str.contains(
    r"\bAMG\s+GT\s+(?:43|53|63)\b",
    regex=True,
    na=False
)

mercedes.loc[
    gt_four_door,
    "apex_model"
] = "AMG GT 4-Door"

# AMG GT sports-car family
gt_sports_car = (
    model_text.str.contains(r"\bAMG\s+GT\b", regex=True, na=False)
    & ~gt_four_door
    & ~model_text.str.contains(r"\bSLS\b", regex=True, na=False)
)

mercedes.loc[
    gt_sports_car,
    "apex_model"
] = "AMG GT Sports Car"

In [25]:
mercedes_performance = mercedes[
    mercedes["apex_model"].notna()
].copy()

mercedes_performance["apex_model"].value_counts()

apex_model
C63                  342
AMG GT 4-Door        314
AMG GT Sports Car    297
E63                  220
Name: count, dtype: int64

In [26]:
mercedes_summary = (
    mercedes_performance
    .groupby("apex_model")
    .agg(
        listings=("model", "size"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_price=("price", "median"),
        median_mileage=("mileage", "median")
    )
    .sort_index()
)

mercedes_summary

,listings,earliest_year,latest_year,median_price,median_mileage
apex_model,,,,,
AMG GT 4-Door,314,2019,2023,102801.5,18593.0
AMG GT Sports Car,297,2016,2021,104990.0,15023.0
C63,342,2008,2021,59000.0,35675.5
E63,220,2007,2021,72528.5,44035.5


In [27]:
mercedes_year_counts = (
    mercedes_performance
    .groupby(["apex_model", "year"])
    .size()
    .unstack(fill_value=0)
)

mercedes_year_counts

year,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
apex_model,,,,,,,,,,,,,,,,,
AMG GT 4-Door,0,0,0,0,0,0,0,0,0,0,0,0,107,92,50,63,2
AMG GT Sports Car,0,0,0,0,0,0,0,0,0,50,32,85,32,71,27,0,0
C63,0,1,3,5,6,7,4,7,10,27,82,60,46,56,28,0,0
E63,5,5,4,6,2,2,5,32,24,0,0,40,45,28,22,0,0


## Porsche Performance Model Coverage

Inspect Porsche model naming conventions and determine whether the dataset contains sufficient 911, Cayman/718, and Boxster listings for analysis.

In [28]:
porsche_models = (
    df.loc[df["manufacturer"] == "Porsche", "model"]
      .value_counts()
)

porsche_models.head(120)

model
Macan Base                    986
Cayenne Base                  709
Macan S                       693
911 Carrera                   425
911 Carrera S                 395
                             ... 
911 Turbo 3.6                   8
Panamera Hybrid S               8
Taycan Cross Turismo Turbo      8
Boxster GTS                     7
912                             7
Name: count, Length: 120, dtype: int64

In [29]:
porsche_target_names = (
    df.loc[
        (df["manufacturer"] == "Porsche") &
        (
            df["model"].str.contains(
                r"\b911\b|\b718\b|\bCayman\b|\bBoxster\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

porsche_target_names.head(120)

model
911 Carrera                    425
911 Carrera S                  395
911 Turbo S                    319
911 Turbo                      275
911 GT3                        254
                              ... 
911 CARRERA S                    1
911 GT2                          1
911 Carrera 4 Black Edition      1
718 Spyder Spyder                1
718 Spyder 2DR SPYDER            1
Name: count, Length: 119, dtype: int64

## Porsche Model Normalization

Porsche listings contain many trim-level descriptions. For the initial feasibility analysis, vehicles are grouped into broader enthusiast model families: 911, Cayman/718 Coupe, and Boxster/718 Roadster.

In [30]:
porsche = df[df["manufacturer"] == "Porsche"].copy()

model_text = porsche["model"].str.upper()

porsche["apex_model"] = pd.NA

# Porsche 911 family
porsche.loc[
    model_text.str.contains(r"\b911\b", regex=True, na=False),
    "apex_model"
] = "911"

# Cayman and 718 Cayman coupes
porsche.loc[
    model_text.str.contains(r"\bCAYMAN\b", regex=True, na=False),
    "apex_model"
] = "Cayman / 718 Coupe"

# Boxster and 718 Boxster roadsters
porsche.loc[
    model_text.str.contains(r"\bBOXSTER\b", regex=True, na=False),
    "apex_model"
] = "Boxster / 718 Roadster"

# 718 Spyder is a roadster even though "Boxster" may not appear in the listing
porsche.loc[
    model_text.str.contains(r"\b718\s+SPYDER\b", regex=True, na=False),
    "apex_model"
] = "Boxster / 718 Roadster"

In [31]:
porsche_performance = porsche[
    porsche["apex_model"].notna()
].copy()

porsche_performance["apex_model"].value_counts()

apex_model
911                       3023
Boxster / 718 Roadster     627
Cayman / 718 Coupe         427
Name: count, dtype: int64

In [32]:
porsche_summary = (
    porsche_performance
    .groupby("apex_model")
    .agg(
        listings=("model", "size"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_price=("price", "median"),
        median_mileage=("mileage", "median")
    )
    .sort_index()
)

porsche_summary

,listings,earliest_year,latest_year,median_price,median_mileage
apex_model,,,,,
911,3023,1965,2023,125420.0,18832.0
Boxster / 718 Roadster,627,1997,2023,42995.0,36847.0
Cayman / 718 Coupe,427,2006,2023,59995.0,23219.0


In [33]:
porsche_year_counts = (
    porsche_performance
    .groupby(["apex_model", "year"])
    .size()
    .unstack(fill_value=0)
)

porsche_year_counts

year,1965,1967,1969,1970,1971,1972,1973,1974,1975,1976,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
apex_model,,,,,,,,,,,,,,,,,,,,,
911,2,1,2,2,4,2,9,3,3,2,...,142,142,134,254,258,259,203,204,371,52
Boxster / 718 Roadster,0,0,0,0,0,0,0,0,0,0,...,22,31,28,29,55,42,15,39,52,6
Cayman / 718 Coupe,0,0,0,0,0,0,0,0,0,0,...,33,22,40,31,64,30,22,32,36,16


## Audi RS Performance Model Coverage

Inspect Audi model naming conventions and determine whether the dataset contains sufficient RS3, RS5, RS6, and RS7 listings for enthusiast-market analysis.

In [35]:
audi_models = (
    df.loc[df["manufacturer"] == "Audi", "model"]
      .value_counts()
)

audi_models.head(120)

model
Q5 2.0T Premium Plus            1087
Q5 2.0T Premium                  499
Q5 45 S line Premium Plus        405
Q7 3.0T Premium Plus             378
A5 2.0T Premium                  378
                                ... 
A4 2.0T Prestige                  32
Q5 3.0 TDI Premium Plus           32
A4 45 S line quattro Premium      30
A4 2.0T quattro                   30
A8 L 4.0T Sport                   30
Name: count, Length: 120, dtype: int64

In [36]:
audi_rs_names = (
    df.loc[
        (df["manufacturer"] == "Audi") &
        (
            df["model"].str.contains(
                r"\bRS\s*3\b|\bRS\s*5\b|\bRS\s*6\b|\bRS\s*7\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

audi_rs_names.head(120)

model
RS 5 2.9T                                                               153
RS 7 4.0T quattro                                                        60
RS 3 2.5T                                                                56
RS 5 4.2                                                                 51
RS 7 4.0T Prestige                                                       41
RS 7 4.0T Performance Prestige                                           14
RS6 Quattro                                                               6
RS 7 PRESTIGE                                                             6
RS 7 4.0T                                                                 6
RS 7 4.0T PRESTIGE                                                        4
RS 5                                                                      2
RS 7                                                                      2
RS 5 DYNAMIC PLUS, ALU OPTIC CARBON FIBER, DRIVER ASSISTANCE, LOADED      1
RS 5 2

## Audi RS Model Normalization

Audi RS listings use several naming conventions and trim descriptions. Listings are normalized into RS3, RS5, RS6, and RS7 model families for feasibility analysis.

In [37]:
audi = df[df["manufacturer"] == "Audi"].copy()

model_text = audi["model"].str.upper()

audi["apex_model"] = pd.NA

audi.loc[
    model_text.str.contains(r"\bRS\s*3\b", regex=True, na=False),
    "apex_model"
] = "RS3"

audi.loc[
    model_text.str.contains(r"\bRS\s*5\b", regex=True, na=False),
    "apex_model"
] = "RS5"

audi.loc[
    model_text.str.contains(r"\bRS\s*6\b", regex=True, na=False),
    "apex_model"
] = "RS6"

audi.loc[
    model_text.str.contains(r"\bRS\s*7\b", regex=True, na=False),
    "apex_model"
] = "RS7"

In [38]:
audi_performance = audi[
    audi["apex_model"].notna()
].copy()

audi_performance["apex_model"].value_counts()

apex_model
RS5    208
RS7    137
RS3     58
RS6      6
Name: count, dtype: int64

In [39]:
audi_summary = (
    audi_performance
    .groupby("apex_model")
    .agg(
        listings=("model", "size"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_price=("price", "median"),
        median_mileage=("mileage", "median")
    )
    .sort_index()
)

audi_summary

,listings,earliest_year,latest_year,median_price,median_mileage
apex_model,,,,,
RS3,58,2017,2023,54775.0,29784.0
RS5,208,2013,2023,58934.0,30390.0
RS6,6,2003,2003,18447.5,106527.5
RS7,137,2014,2023,75000.0,27607.0


## Chevrolet Corvette Model Coverage

Inspect Chevrolet Corvette naming conventions and determine whether the dataset contains sufficient listings across Corvette generations for enthusiast-market analysis.

In [41]:
corvette_names = (
    df.loc[
        (df["manufacturer"] == "Chevrolet") &
        (
            df["model"].str.contains(
                r"\bCorvette\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

corvette_names.head(120)

model
Corvette Base                539
Corvette Stingray w/3LT      388
Corvette                     355
Corvette Stingray w/2LT      329
Corvette Stingray            281
                            ... 
Corvette LT1                   1
Corvette BASE                  1
Corvette 2LT Z51 PKG           1
Corvette Coupe 3Lt             1
Corvette Stingray Z51 2LT      1
Name: count, Length: 70, dtype: int64

In [42]:
corvette = df[
    (df["manufacturer"] == "Chevrolet") &
    (
        df["model"].str.contains(
            r"\bCorvette\b",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

In [43]:
corvette_summary = pd.DataFrame({
    "listings": [len(corvette)],
    "earliest_year": [corvette["year"].min()],
    "latest_year": [corvette["year"].max()],
    "median_price": [corvette["price"].median()],
    "median_mileage": [corvette["mileage"].median()]
})

corvette_summary

,listings,earliest_year,latest_year,median_price,median_mileage
0,3128,1953,2023,56276.0,18070.0


In [44]:
corvette["year"].value_counts().sort_index()

year
1953      1
1954      2
1955      4
1956      2
1957      3
       ... 
2019    223
2020    178
2021    238
2022    263
2023    264
Name: count, Length: 70, dtype: int64

## Corvette Generation Mapping

Corvette listings span several decades and fundamentally different vehicle generations. For the primary market analysis, modern Corvettes are grouped into C5, C6, C7, and C8 generations to enable more meaningful comparisons of pricing, mileage, and depreciation behavior.

In [45]:
def corvette_generation(year):
    if 1997 <= year <= 2004:
        return "C5"
    elif 2005 <= year <= 2013:
        return "C6"
    elif 2014 <= year <= 2019:
        return "C7"
    elif year >= 2020:
        return "C8"
    else:
        return "Pre-C5"

corvette["generation"] = corvette["year"].apply(corvette_generation)

In [46]:
corvette["generation"].value_counts()

generation
C7        1006
C8         943
C6         532
Pre-C5     360
C5         287
Name: count, dtype: int64

In [47]:
corvette_generation_summary = (
    corvette[
        corvette["generation"] != "Pre-C5"
    ]
    .groupby("generation")
    .agg(
        listings=("model", "size"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_price=("price", "median"),
        median_mileage=("mileage", "median")
    )
    .reindex(["C5", "C6", "C7", "C8"])
)

corvette_generation_summary

,listings,earliest_year,latest_year,median_price,median_mileage
generation,,,,,
C5,287,1997,2004,21997.0,51760.0
C6,532,2005,2013,31523.5,40494.0
C7,1006,2014,2019,56955.5,20234.0
C8,943,2020,2023,92000.0,2555.0


In [48]:
pd.crosstab(
    corvette["generation"],
    corvette["transmission"].apply(classify_transmission),
    margins=True
)

transmission,Automatic,Manual,Other/Unknown,Unknown,All
generation,,,,,
C5,180,101,0,6,287
C6,303,217,5,7,532
C7,771,214,4,17,1006
C8,650,15,247,31,943
Pre-C5,192,129,6,33,360
All,2096,676,262,94,3128


### Corvette Transmission Data Quality Check

The C8 Corvette was not offered with a conventional manual transmission, yet some listings are classified as manual. These records are inspected to determine whether the discrepancy results from inconsistent marketplace transmission descriptions or data-quality issues.

In [50]:
c8 = corvette[corvette["generation"] == "C8"].copy()

c8["transmission_group"] = (
    c8["transmission"].apply(classify_transmission)
)

c8.loc[
    c8["transmission_group"] == "Manual",
    ["year", "model", "transmission", "price", "mileage"]
]

,year,model,transmission,price,mileage
100169,2022,Corvette Stingray w/1LT,8-Speed Manual,92500.0,7965.0
100284,2022,Corvette Stingray w/2LT,8-Speed Manual,105000.0,24.0
100710,2023,Corvette Stingray w/2LT,8-Speed Manual,99650.0,275.0
100734,2023,Corvette Stingray w/2LT,8-Speed Manual,89900.0,1097.0
100881,2023,Corvette Stingray w/1LT,8-Speed Manual,94777.0,2063.0
100920,2022,Corvette Stingray w/2LT,8-Speed Manual,119500.0,1093.0
101371,2023,Corvette Stingray w/3LT,8-Speed Manual,107990.0,30.0
101682,2022,Corvette Stingray w/1LT,8-Speed Manual,105750.0,48.0
101741,2023,Corvette Stingray w/1LT,Manual,117458.0,49.0
102191,2022,Corvette Stingray w/3LT,8-Speed Manual,97777.0,350.0


In [51]:
c8.loc[
    c8["transmission_group"] == "Other/Unknown",
    "transmission"
].value_counts()

transmission
8-Speed                           220
Transmission w/Dual Shift Mode     22
8-Speed Double Clutch               3
NOT SPECIFIED                       2
Name: count, dtype: int64

In [52]:
c8["transmission"].value_counts().head(30)

transmission
8-Speed Automatic with Auto-Shift    348
Automatic                            229
8-Speed                              220
8-Speed Automatic                     37
8-Speed A/T                           31
Transmission w/Dual Shift Mode        22
8-Speed Manual                        13
A/T                                    4
8-Speed Double Clutch                  3
NOT SPECIFIED                          2
Manual                                 2
Automatic 8-Speed                      1
Name: count, dtype: int64

### C8 Transmission Data Quality Finding

The raw dataset contains inconsistent transmission labels for C8 Corvette listings, including values such as "8-Speed Manual" and "Manual." Because these values conflict with the expected transmission configuration for the C8 generation, the raw field will be preserved and a separate normalized transmission field and data-quality flag will be created during the cleaning phase.

## Toyota Supra Model Coverage

Inspect Toyota Supra naming conventions and determine whether the dataset contains sufficient listings for modern and earlier-generation Supra analysis.

In [53]:
supra_names = (
    df.loc[
        (df["manufacturer"] == "Toyota") &
        (
            df["model"].str.contains(
                r"\bSupra\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

supra_names.head(100)

model
Supra 3.0                                                   111
Supra 3.0 Premium                                            55
Supra 2.0                                                    51
Supra Turbo                                                  23
Supra A91 Edition                                            13
Supra 2                                                      11
Supra                                                        10
Supra A91-CF Edition                                          7
Supra A91-MT Edition                                          5
Supra 3.0 Premium Launch Edition                              5
Supra Base                                                    4
Supra 3.0 PREMIU                                              3
Supra 3.0 Premiu                                              3
Celica Supra                                                  1
Supra NULL                                                    1
Supra LAUNCH EDITION BLND SPT NAV 

In [54]:
supra = df[
    (df["manufacturer"] == "Toyota") &
    (
        df["model"].str.contains(
            r"\bSupra\b",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

In [55]:
supra_summary = pd.DataFrame({
    "listings": [len(supra)],
    "earliest_year": [supra["year"].min()],
    "latest_year": [supra["year"].max()],
    "median_price": [supra["price"].median()],
    "median_mileage": [supra["mileage"].median()]
})

supra_summary

,listings,earliest_year,latest_year,median_price,median_mileage
0,308,1983,2023,54995.0,11424.0


In [56]:
supra["year"].value_counts().sort_index()

year
1983      1
1987      3
1989      1
1990      1
1992      1
1993      7
1994      8
1995      2
1996      2
1997      8
1998      3
2020     57
2021    110
2022     79
2023     25
Name: count, dtype: int64

In [57]:
pd.crosstab(
    supra["year"],
    supra["transmission"].apply(classify_transmission),
    margins=True
)

transmission,Automatic,Manual,Other/Unknown,Unknown,All
year,,,,,
1983,0,1,0,0,1
1987,2,1,0,0,3
1989,0,1,0,0,1
1990,1,0,0,0,1
1992,0,1,0,0,1
1993,1,3,0,3,7
1994,3,5,0,0,8
1995,0,2,0,0,2
1996,0,1,1,0,2


### Supra Generation Scope

The dataset contains limited observations for older Supra generations but strong coverage for the modern A90/A91 generation. The primary Apex Analytics analysis therefore focuses on 2020–2023 Supra listings, while earlier models are retained only for descriptive context.

In [58]:
supra["generation"] = supra["year"].apply(
    lambda year: "A90/A91" if year >= 2020 else "Pre-A90"
)

supra["generation"].value_counts()

generation
A90/A91    271
Pre-A90     37
Name: count, dtype: int64

In [59]:
supra_generation_summary = (
    supra
    .groupby("generation")
    .agg(
        listings=("model", "size"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_price=("price", "median"),
        median_mileage=("mileage", "median")
    )
)

supra_generation_summary

,listings,earliest_year,latest_year,median_price,median_mileage
generation,,,,,
A90/A91,271,2020,2023,54251.0,9650.0
Pre-A90,37,1983,1998,94999.0,68741.0


## Nissan GT-R Model Coverage

Inspect Nissan GT-R naming conventions and determine whether the dataset contains sufficient R35 GT-R listings for pricing, mileage, and model-year analysis.

In [60]:
gtr_names = (
    df.loc[
        (df["manufacturer"] == "Nissan") &
        (
            df["model"].str.contains(
                r"\bGT[\s-]?R\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

gtr_names.head(100)

model
GT-R Premium                                     108
GT-R Black Edition                                25
GT-R NISMO                                         7
GT-R Premium Dual-clutch 6-Speed Transmission      4
GT-R T-spec                                        3
GT-R Track Edition                                 3
GT-R Pure                                          1
GT-R NISMO Special Edition                         1
GT-R Base                                          1
GT-R PREMIUM-GT1R-T1 RACE DEV. SERIES 4            1
Name: count, dtype: int64

In [61]:
gtr = df[
    (df["manufacturer"] == "Nissan") &
    (
        df["model"].str.contains(
            r"\bGT[\s-]?R\b",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

In [62]:
gtr_summary = pd.DataFrame({
    "listings": [len(gtr)],
    "earliest_year": [gtr["year"].min()],
    "latest_year": [gtr["year"].max()],
    "median_price": [gtr["price"].median()],
    "median_mileage": [gtr["mileage"].median()]
})

gtr_summary

,listings,earliest_year,latest_year,median_price,median_mileage
0,154,2009,2023,94941.5,24451.5


In [63]:
gtr["year"].value_counts().sort_index()

year
2009    10
2010     8
2011     2
2012     6
2013    13
2014    16
2015    24
2016    13
2017    16
2018     4
2019     2
2020    14
2021    14
2023    12
Name: count, dtype: int64

## Cadillac V-Series and Blackwing Model Coverage

Inspect Cadillac performance-model naming conventions and determine whether the dataset contains sufficient CT4-V, CT5-V, and Blackwing listings for enthusiast-market analysis.

In [64]:
blackwing_names = (
    df.loc[
        (df["manufacturer"] == "Cadillac") &
        (
            df["model"].str.contains(
                r"\bBlackwing\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

blackwing_names

model
CT4-V V-Series Blackwing           80
CT5-V V-Series Blackwing           54
CT6-V 4.2L Blackwing Twin Turbo    12
CT5-V Blackwing                     2
CT5-V Blackwing 4dr Sedan           1
CT6-V Blackwing Twin                1
CT4-V Blackwing                     1
Name: count, dtype: int64

In [65]:
cadillac_v_names = (
    df.loc[
        (df["manufacturer"] == "Cadillac") &
        (
            df["model"].str.contains(
                r"\bCT4\b|\bCT5\b",
                case=False,
                na=False,
                regex=True
            )
        ),
        "model"
    ]
    .value_counts()
)

cadillac_v_names.head(100)

model
CT5 Premium Luxury AWD       154
CT4 Premium Luxury           126
CT4 Sport                    123
CT5                          109
CT5 Premium Luxury RWD       105
CT5 Sport AWD                 92
CT4 Luxury                    85
CT4-V V-Series Blackwing      80
CT5 Luxury RWD                69
CT5 Sport RWD                 58
CT5-V V-Series Blackwing      54
CT4                           53
CT5 V-Series                  53
CT4 V-Series                  48
CT5 Luxury AWD                47
CT5-V V-Series                27
CT4-V V-Series                20
CT4-V                         10
CT5-V                          6
CT5 Premium Luxury             5
CT5 Luxury                     4
CT5-V Base                     3
CT5-V Blackwing                2
CT5 Sport                      2
CT5-V 4dr Sdn                  1
CT5-V Blackwing 4dr Sedan      1
CT4-V Blackwing                1
Name: count, dtype: int64

## Cadillac Blackwing Model Normalization

Cadillac performance listings use multiple naming conventions for CT4-V and CT5-V Blackwing models. These records are normalized into two enthusiast model families for feasibility analysis.

In [66]:
cadillac = df[df["manufacturer"] == "Cadillac"].copy()

model_text = cadillac["model"].str.upper()

cadillac["apex_model"] = pd.NA

# CT4-V Blackwing
cadillac.loc[
    model_text.str.contains(
        r"\bCT4[\s-]*V\b.*\bBLACKWING\b",
        regex=True,
        na=False
    ),
    "apex_model"
] = "CT4-V Blackwing"

# CT5-V Blackwing
cadillac.loc[
    model_text.str.contains(
        r"\bCT5[\s-]*V\b.*\bBLACKWING\b",
        regex=True,
        na=False
    ),
    "apex_model"
] = "CT5-V Blackwing"

In [67]:
cadillac_performance = cadillac[
    cadillac["apex_model"].notna()
].copy()

cadillac_performance["apex_model"].value_counts()

apex_model
CT4-V Blackwing    81
CT5-V Blackwing    57
Name: count, dtype: int64

In [68]:
cadillac_summary = (
    cadillac_performance
    .groupby("apex_model")
    .agg(
        listings=("model", "size"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        median_price=("price", "median"),
        median_mileage=("mileage", "median")
    )
    .sort_index()
)

cadillac_summary

,listings,earliest_year,latest_year,median_price,median_mileage
apex_model,,,,,
CT4-V Blackwing,81,2022,2023,68352.0,3829.0
CT5-V Blackwing,57,2022,2023,109999.0,1691.0


In [69]:
cadillac_performance["transmission_group"] = (
    cadillac_performance["transmission"]
    .apply(classify_transmission)
)

pd.crosstab(
    cadillac_performance["apex_model"],
    cadillac_performance["transmission_group"],
    margins=True
)

transmission_group,Automatic,Manual,Other/Unknown,Unknown,All
apex_model,,,,,
CT4-V Blackwing,56,24,0,1,81
CT5-V Blackwing,28,28,1,0,57
All,84,52,1,1,138


## Phase 0 — Master Feasibility Summary

The following table consolidates the enthusiast vehicle segments evaluated during the dataset audit. Vehicle segments are classified according to available sample size before the final project scope is selected.

- **Core candidate:** 200+ listings
- **Secondary candidate:** 50–199 listings
- **Exclude:** fewer than 50 listings

Sample size is used as an initial feasibility criterion only. Final inclusion will also consider analytical relevance, comparability, and project scope.

In [70]:
def summarize_segment(brand, segment, data):
    return {
        "brand": brand,
        "segment": segment,
        "listings": len(data),
        "earliest_year": int(data["year"].min()),
        "latest_year": int(data["year"].max()),
        "median_price": data["price"].median(),
        "median_mileage": data["mileage"].median()
    }

In [71]:
feasibility_rows = []

# BMW M
for model in ["M2", "M3", "M4", "M5"]:
    subset = bmw_performance[
        bmw_performance["apex_model"] == model
    ]
    
    feasibility_rows.append(
        summarize_segment("BMW", model, subset)
    )


# Mercedes-AMG
for model in ["C63", "E63", "AMG GT Sports Car", "AMG GT 4-Door"]:
    subset = mercedes_performance[
        mercedes_performance["apex_model"] == model
    ]
    
    feasibility_rows.append(
        summarize_segment("Mercedes-AMG", model, subset)
    )


# Porsche
for model in [
    "911",
    "Cayman / 718 Coupe",
    "Boxster / 718 Roadster"
]:
    subset = porsche_performance[
        porsche_performance["apex_model"] == model
    ]
    
    feasibility_rows.append(
        summarize_segment("Porsche", model, subset)
    )


# Audi RS
for model in ["RS3", "RS5", "RS6", "RS7"]:
    subset = audi_performance[
        audi_performance["apex_model"] == model
    ]
    
    feasibility_rows.append(
        summarize_segment("Audi", model, subset)
    )


# Corvette generations
for generation in ["C5", "C6", "C7", "C8"]:
    subset = corvette[
        corvette["generation"] == generation
    ]
    
    feasibility_rows.append(
        summarize_segment(
            "Chevrolet",
            f"Corvette {generation}",
            subset
        )
    )


# Modern Toyota Supra
supra_modern = supra[
    supra["generation"] == "A90/A91"
]

feasibility_rows.append(
    summarize_segment(
        "Toyota",
        "Supra A90/A91",
        supra_modern
    )
)


# Nissan GT-R
feasibility_rows.append(
    summarize_segment(
        "Nissan",
        "GT-R (R35)",
        gtr
    )
)


# Cadillac Blackwing
for model in ["CT4-V Blackwing", "CT5-V Blackwing"]:
    subset = cadillac_performance[
        cadillac_performance["apex_model"] == model
    ]
    
    feasibility_rows.append(
        summarize_segment("Cadillac", model, subset)
    )

In [72]:
master_feasibility = pd.DataFrame(feasibility_rows)

In [73]:
def feasibility_status(listings):
    if listings >= 200:
        return "Core candidate"
    elif listings >= 50:
        return "Secondary candidate"
    else:
        return "Exclude"


master_feasibility["status"] = (
    master_feasibility["listings"]
    .apply(feasibility_status)
)

In [74]:
master_feasibility["median_price"] = (
    master_feasibility["median_price"].round(0)
)

master_feasibility["median_mileage"] = (
    master_feasibility["median_mileage"].round(0)
)

master_feasibility = (
    master_feasibility
    .sort_values(
        ["listings"],
        ascending=False
    )
    .reset_index(drop=True)
)

master_feasibility

,brand,segment,listings,earliest_year,latest_year,median_price,median_mileage,status
0,Porsche,911,3023,1965,2023,125420.0,18832.0,Core candidate
1,Chevrolet,Corvette C7,1006,2014,2019,56956.0,20234.0,Core candidate
2,Chevrolet,Corvette C8,943,2020,2023,92000.0,2555.0,Core candidate
3,Porsche,Boxster / 718 Roadster,627,1997,2023,42995.0,36847.0,Core candidate
4,BMW,M4,565,2015,2023,63677.0,20008.0,Core candidate
5,Chevrolet,Corvette C6,532,2005,2013,31524.0,40494.0,Core candidate
6,BMW,M3,528,1988,2023,48900.0,50470.0,Core candidate
7,Porsche,Cayman / 718 Coupe,427,2006,2023,59995.0,23219.0,Core candidate
8,BMW,M5,396,1988,2023,73934.0,32932.0,Core candidate
9,Mercedes-AMG,C63,342,2008,2021,59000.0,35676.0,Core candidate


## Final Project Scope

Based on sample size, analytical relevance, and comparability, the candidate vehicles are divided into three groups:

- **Core:** Primary vehicles used for detailed market, pricing, generation, and statistical analysis.
- **Supporting:** Vehicles retained for dashboard comparisons or focused case studies but not necessarily used in every statistical analysis.
- **Exclude:** Vehicles with insufficient observations for reliable analysis.

Final inclusion is based on both data availability and the analytical purpose of Apex Analytics rather than sample size alone.

In [75]:
core_segments = {
    ("Porsche", "911"),
    ("Porsche", "Cayman / 718 Coupe"),

    ("BMW", "M3"),
    ("BMW", "M4"),

    ("Mercedes-AMG", "C63"),

    ("Audi", "RS5"),

    ("Chevrolet", "Corvette C6"),
    ("Chevrolet", "Corvette C7"),
    ("Chevrolet", "Corvette C8"),

    ("Toyota", "Supra A90/A91")
}


supporting_segments = {
    ("BMW", "M2"),
    ("BMW", "M5"),

    ("Mercedes-AMG", "E63"),
    ("Mercedes-AMG", "AMG GT Sports Car"),
    ("Mercedes-AMG", "AMG GT 4-Door"),

    ("Porsche", "Boxster / 718 Roadster"),

    ("Audi", "RS3"),
    ("Audi", "RS7"),

    ("Chevrolet", "Corvette C5"),

    ("Nissan", "GT-R (R35)"),

    ("Cadillac", "CT4-V Blackwing"),
    ("Cadillac", "CT5-V Blackwing")
}

In [76]:
def assign_final_scope(row):
    key = (row["brand"], row["segment"])

    if key in core_segments:
        return "Core"

    if key in supporting_segments:
        return "Supporting"

    return "Exclude"


master_feasibility["final_scope"] = (
    master_feasibility.apply(assign_final_scope, axis=1)
)

In [77]:
scope_order = pd.CategoricalDtype(
    categories=["Core", "Supporting", "Exclude"],
    ordered=True
)

master_feasibility["final_scope"] = (
    master_feasibility["final_scope"].astype(scope_order)
)

In [78]:
final_scope_table = (
    master_feasibility[
        [
            "brand",
            "segment",
            "listings",
            "earliest_year",
            "latest_year",
            "median_price",
            "median_mileage",
            "final_scope"
        ]
    ]
    .sort_values(
        ["final_scope", "listings"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

final_scope_table

,brand,segment,listings,earliest_year,latest_year,median_price,median_mileage,final_scope
0,Porsche,911,3023,1965,2023,125420.0,18832.0,Core
1,Chevrolet,Corvette C7,1006,2014,2019,56956.0,20234.0,Core
2,Chevrolet,Corvette C8,943,2020,2023,92000.0,2555.0,Core
3,BMW,M4,565,2015,2023,63677.0,20008.0,Core
4,Chevrolet,Corvette C6,532,2005,2013,31524.0,40494.0,Core
5,BMW,M3,528,1988,2023,48900.0,50470.0,Core
6,Porsche,Cayman / 718 Coupe,427,2006,2023,59995.0,23219.0,Core
7,Mercedes-AMG,C63,342,2008,2021,59000.0,35676.0,Core
8,Toyota,Supra A90/A91,271,2020,2023,54251.0,9650.0,Core
9,Audi,RS5,208,2013,2023,58934.0,30390.0,Core


In [79]:
final_scope_table.groupby(
    "final_scope",
    observed=True
)["listings"].agg(["count", "sum"])

,count,sum
final_scope,,
Core,10,7845
Supporting,12,2785
Exclude,1,6


## Phase 0 Conclusion

The feasibility audit confirms that the dataset provides sufficient coverage to support Apex Analytics as an enthusiast used-car market intelligence project.

The final scope contains 10 core vehicle segments representing 7,845 listings and 12 supporting segments representing an additional 2,785 listings. Audi RS6 was excluded because the dataset contains only six observations.

The core analysis will focus on:

- Porsche 911
- Porsche Cayman / 718 Coupe
- BMW M3
- BMW M4
- Mercedes-AMG C63
- Audi RS5
- Chevrolet Corvette C6
- Chevrolet Corvette C7
- Chevrolet Corvette C8
- Toyota Supra A90/A91

Supporting vehicles will be retained for dashboard comparisons and focused case studies where appropriate.

Several data-quality issues were identified during the audit, including inconsistent transmission descriptions and model naming conventions. In particular, some C8 Corvette listings contain transmission labels that conflict with the expected vehicle configuration. Raw values will therefore be preserved while separate normalized fields and quality flags are created during the cleaning phase.

The next phase will focus on cleaning, normalization, generation mapping, and preparation of an analysis-ready enthusiast vehicle dataset.